# MGnify List vs. Detail endpoints

The [MGnify API](https://www.ebi.ac.uk/metagenomics/api/v2) has 2 types of endpoints: 

1. **list** endpoints which return a (paginated) list of records (dicts) in brief from a MGnify resource
2. **detail** endpoints which return a single record (dict) in lots of detail

The list endpoints can accept different search params to filter down the list (e.g., "search", "biome_lineage", "page_size"). In contrast, the detail endpoints only accept a single accession/id. 

In MGni.py, the MGnifier's that correspond to 
1. list endpoints are plural e.g. `MG.samples`
2. detail endpoints are singular e.g. `MG.sample`

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
from mgnipy import MGnipy

# init client
MG = MGnipy(cache_dir=None)

# check out the endpoints
print(MG.list_resources())

['analyses', 'analysis', 'assemblies', 'assembly', 'genomes', 'genome', 'publications', 'publication', 'samples', 'sample', 'studies', 'study', 'runs', 'run', 'biomes', 'biome', 'miscellaneous', 'catalogues', 'catalogue', 'private_studies']


From `list_resources()` we see plural vs. singular terms e.g. `analyses` vs. `analysis`. 

The plural attributes are list and singular are detail endpoints:

In [2]:
# accessing a list endpoint
studies_list = MG.studies(search='diabetes')
print(studies_list)
# now getting the list
with MG: 
    studies_list.get() # or .get_all()

display(studies_list.search_results.to_pandas().head())

# accessing a detail endpoint
a_study_detail = MG.study("MGYS00006805")
print(a_study_detail)
# now getting the record
with MG: 
    a_study_detail.get()

display(a_study_detail.search_results.to_pandas())

<class 'mgnipy.V2.proxies.studies.Studies'> for 'studies' resource
- Endpoint: 'mgnipy.emgapi_v2_client.api.studies.list_mgnify_studies'
- Params: {'search': 'diabetes'}
- Child resource: 'study' 


,accession,ena_accessions,title,biome,updated_at,metadata
0,MGYS00006805,"[PRJEB63337, ERP148499]",EMG produced TPA metagenomics assembly of PRJD...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-01T19:06:56.381000+00:00,{}
1,MGYS00010387,"[PRJDB9649, DRP008416]",Fecal microbiota transplantation alleviates di...,None,2026-05-28T15:46:48.985000+00:00,{}
2,MGYS00010379,"[SRP387956, PRJNA862077]",gut metagenome and type-1 diabetes,None,2026-05-28T15:46:49.164000+00:00,{}
3,MGYS00005377,"[ERP114158, PRJEB31588]",EMG produced TPA metagenomics assembly of the ...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:46:57.525000+00:00,{}
4,MGYS00005378,"[SRP056054, PRJNA231909]","A prospective, longitudinal analysis of the de...","{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:46:57.527000+00:00,{}


<class 'mgnipy.V2.proxies.studies.StudyDetail'> for 'MGYS00006805'
- Endpoint: 'mgnipy.emgapi_v2_client.api.studies.get_mgnify_study'
- Supported relationships: ['samples', 'analyses', 'publications'] 


,accession,ena_accessions,title,biome,updated_at,metadata,downloads,first_accession
0,MGYS00006805,"[PRJEB63337, ERP148499]",EMG produced TPA metagenomics assembly of PRJD...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-01T19:06:56.381000+00:00,{},"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP148499


## From list to detailed list

After getting a list of records from a list endpoint, one can beef up the list with additional metadata by using the "child" detail endpoint: e.g. `samples` to `sample`, `assemblies` to `assembly`, etc

there is a method `.enrich_details()` for MGnifyList's that help with this which will iteratively get the list of MGnifyDetails. 

In [3]:
# populating the list 
with MG: 
    studies_list.enrich_details(limit=3) #can set to None to get all

# checking out the detailed metdata
studies_list.metadata.to_pandas(expand_nested_dicts=True)

Enriching study details:  25%|██▌       | 3/12 [00:00<00:01,  7.29it/s]


,accession,ena_accessions,title,updated_at,downloads,first_accession,biome__biome_name,biome__lineage
0,MGYS00006805,"[PRJEB63337, ERP148499]",EMG produced TPA metagenomics assembly of PRJD...,2026-05-01T19:06:56.381000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP148499,Fecal,root:Host-associated:Human:Digestive system:La...
1,MGYS00010387,"[PRJDB9649, DRP008416]",Fecal microbiota transplantation alleviates di...,2026-05-28T15:46:48.985000+00:00,[],DRP008416,NaN,NaN
2,MGYS00010379,"[SRP387956, PRJNA862077]",gut metagenome and type-1 diabetes,2026-05-28T15:46:49.164000+00:00,[],SRP387956,NaN,NaN


For the enriched studies we can get their MGnifyDetail object via indexing:

In [4]:
# e.g. int
a_study_detail = studies_list[0]

# or by accession/id
a_study_detail = studies_list['MGYS00006805']
print(a_study_detail)

<class 'mgnipy.V2.proxies.studies.StudyDetail'> for 'MGYS00006805'
- Endpoint: 'mgnipy.emgapi_v2_client.api.studies.get_mgnify_study'
- Supported relationships: ['samples', 'analyses', 'publications'] 


or get all of the details as a dict:

In [5]:
studies_list.mgnify_details

{'MGYS00006805': <<class 'mgnipy.V2.proxies.studies.StudyDetail'> resource='study', id='MGYS00006805'>,
 'MGYS00010387': <<class 'mgnipy.V2.proxies.studies.StudyDetail'> resource='study', id='MGYS00010387'>,
 'MGYS00010379': <<class 'mgnipy.V2.proxies.studies.StudyDetail'> resource='study', id='MGYS00010379'>}

## from detail to lists

If you noticed from the prints of the `StudyDetail`s above there are "Supported relationships" which link to other MGnifyLists. 

This means that from a study you can get their collection of samples for example. 

In [6]:
a_study_detail.list_relationships()

['samples', 'analyses', 'publications']

when we access the relationship, we will automatically `get` the MGnifyList of `.search_results`

However, if we want to enrich with even more detailed `.metadata` we still do the `.enrich_details`

In [18]:
with MG: #context manager
    # access the samples list for a given study
    a_study_samples = a_study_detail.samples 
    # enrich the samples list further with details
    a_study_samples.enrich_details(limit=4) #can set limit to None to get all

# look at detailed samples list so far
a_study_samples.metadata.to_pandas(expand_nested_dicts=True)

Enriching sample details:   5%|▍         | 4/86 [00:00<00:07, 10.50it/s]


,accession,ena_accessions,sample_title,updated_at,studies,metadata__ph,metadata__age,metadata__lat,metadata__lon,metadata__sex,...,metadata__collection_date_start,metadata__sample_capture_status,metadata__host_growth_conditions,metadata__ncbi_reporting_standard,metadata__related_sample_accession,metadata__taxonomic_classification,metadata__taxonomic_identity_marker,metadata__secondary_sample_accession,metadata__local_environmental_context,metadata__broad_scale_environmental_context
0,SAMD00218068,"[SAMD00218068, DRS234959]",FMT12W-4H,2026-05-01T19:05:04.370000+00:00,"[{'accession': 'MGYS00006805', 'ena_accessions...",,,24.7,113.6,,...,2018-01-01,,,MIMS.me.human-gut,,,,DRS234959,gut,human
1,SAMD00218137,"[SAMD00218137, DRS235028]",FMT4W-17H,2026-05-01T19:05:05.804000+00:00,"[{'accession': 'MGYS00006805', 'ena_accessions...",,,24.7,113.6,,...,2018-01-01,,,MIMS.me.human-gut,,,,DRS235028,gut,human
2,SAMD00218113,"[DRS235004, SAMD00218113]",FMTD0-8H,2026-05-01T19:05:07.034000+00:00,"[{'accession': 'MGYS00006805', 'ena_accessions...",,,24.7,113.6,,...,2018-01-01,,,MIMS.me.human-gut,,,,DRS235004,gut,human
3,SAMD00218093,"[SAMD00218093, DRS234984]",FMT8W-19H,2026-05-01T19:05:08.126000+00:00,"[{'accession': 'MGYS00006805', 'ena_accessions...",,,24.7,113.6,,...,2018-01-01,,,MIMS.me.human-gut,,,,DRS234984,gut,human
